# 🏆 Capstone Project: Starter Notebook
### Forecasting agricultural market prices for *Region B* (beans & potato)

Read **`PROJECT_BRIEF.md`** first (goals, deliverables, rules) and **`RUBRIC.md`** (how you will be assessed).
This notebook gives you the *scaffolding*: a leak-proof evaluation harness, an example model, and clearly marked **TODO** blocks. Everything outside the TODO blocks is provided: read it, don't fight it.

> 🎯 **You are graded on the quality of your reasoning, not on the lowest error.** A modest model with an honest, leak-free evaluation beats a flashy one with a hidden leak.

| Section | What you do |
|---|---|
| 0 | Setup (provided) |
| 1 | **Audit** the raw data |
| 2 | **Clean** it (write `clean`) |
| 3 | **Explore & discover** structure (lags, non-linearity, structural change) |
| 4 | Evaluation protocol & the harness (provided) |
| 5 | **Build models** (baselines, statistical, ML; at least 5) |
| 6 | **Compare on the development period** and select |
| 7 | **One-shot** test on the 2024 hold-out (locked until you're ready) |
| 8 | **Interpret** and write the model card |

## 0 · Setup (provided)

In [1]:
import sys; sys.path.append(".."); sys.path.append(".")            # course_utils.py is one folder up; project_utils.py is here
import warnings
import statsmodels.tools.sm_exceptions
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course_utils import *            # metrics, fourier_terms, plotting helpers, leakage_check, ...
from project_utils import *           # run_walk_forward, add_blend, scoreboard, empirical_interval, coverage, extend, HORIZONS

print("Horizons:", HORIZONS)

Horizons: (1, 4, 8, 12)


## 1 · Audit the raw data

We load **only the training file** (2013–2023). The 2024 hold-out is locked until Section 7: *do not open, plot or peek at it before then.*
Read `data/DATA_DICTIONARY.md` for the columns.

**TODO 1**: Audit: how many rows? duplicates? missing values? missing *weeks*? suspicious prices (typos)? Which column would be **leakage** if used as a predictor, and why?

In [2]:
raw = pd.read_csv("data/project_market_train.csv", parse_dates=["date"])
print(raw.shape); display(raw.head())

# TODO 1: audit the data (duplicates, NaNs, missing weeks per commodity, outliers, ranges)
# YOUR CODE HERE

(1140, 8)


,date,commodity,price,arrivals_tonnes,rainfall_mm,temp_c,exchange_rate,festival_flag
0,2013-01-06,beans,118.12,131.9,34.8,22.7,99.03,0
1,2013-01-13,beans,120.21,109.7,38.4,23.5,99.39,0
2,2013-01-20,beans,121.20,106.6,4.8,23.3,99.15,0
3,2013-01-27,beans,124.91,101.9,35.5,22.7,100.54,0
4,2013-02-03,beans,121.54,104.6,66.3,22.4,99.86,0


✍️ **Audit findings** (what is wrong with the data, and what will you do about each problem?):

- …

## 2 · Clean the data

**TODO 2**: Complete `clean(d)`: it receives the raw rows of *one* commodity and must return a **regular weekly DataFrame** (index `W-SUN`, no missing weeks, no NaNs, no duplicate dates).
Remember Lesson 2: fix outliers *before* imputing; prefer **causal** fills (forward-fill) for price; justify anything else; make sure nothing in your cleaning uses **future** information.

In [3]:
def clean(d):
    """Raw rows of ONE commodity -> clean regular weekly DataFrame."""
    d = d.drop(columns="commodity").set_index("date").sort_index()
    # TODO 2: duplicates · typos -> NaN · reindex to every week · fill price (causally) · fill rainfall outage · fill other columns
    # YOUR CODE HERE
    return d

data = {c: clean(g) for c, g in raw.groupby("commodity")}
for c, d in data.items():
    print(f"{c:<7} {len(d)} weeks | NaNs left: {int(d.isna().sum().sum())} | duplicated dates: {int(d.index.duplicated().sum())}")

beans   571 weeks | NaNs left: 3 | duplicated dates: 2
potato  569 weeks | NaNs left: 3 | duplicated dates: 0


## 3 · Explore & discover

This market is *not* the one from the lessons. Before modelling, find out how it works. Use **only the training data**.

**TODO 3**: Investigate and document (with a plot or number for each):
1. Trend and seasonality of each commodity (STL; seasonal strength; which months are dear/cheap?).
2. Does **rainfall lead price**? By how many weeks? (hint: correlate *changes*, try lags 0…12). What does that imply for which horizons can use rain?
3. Do **exchange rate** or **temperature** matter? Is any effect **non-linear / threshold-like**?
4. Has the market's **structure changed** over time (e.g. seasonal pattern before vs after some date)?
5. Stationarity: ADF/KPSS and differencing needed?

In [4]:
# TODO 3: your exploration (add as many cells as you need)
# YOUR CODE HERE

✍️ **What I discovered** (each claim backed by a plot/number above):

1. …
2. …
3. …
4. …

## 4 · The evaluation protocol and the leak-proof harness (provided)

| | |
|---|---|
| **Horizons** | 1, 4, 8, 12 weeks |
| **Development period** | forecast origins from 2022-01-01 to the end of 2023, every 4 weeks, expanding window, refit at each origin. **All model selection happens here.** |
| **Test period** | the 2024 hold-out, evaluated **once** at the end (Section 7) |
| **Metric** | MAE (price units) + skill vs naive (`1 − MAE / MAE_naive`); report RMSE too if you like |

### The harness
A model is a function `model(train, h, future) -> float`:
- `train`: a DataFrame with **only the weeks up to the origin** (the harness slices it; you physically cannot see later rows),
- `h`: horizon in weeks,
- `future`: the next `h` weeks of calendar-known columns (`festival_flag`),
- returns the **price forecast for week origin + h**.

Helpers: `extend(train, future)` appends the future rows (empty except the festival flag) so you can compute features for the *target* week; features must only use data ≥ `h` weeks old.
Study the example below, then write your own.

In [5]:
def model_naive(train, h, fut):
    return train["price"].iloc[-1]                                   # tomorrow = today

MODELS = {"naive": model_naive}                                      # you will add more models here

## 5 · Build models

**TODO 5**: Add **at least five** models to `MODELS` (name → function), including:
1. the **naive** baseline (given) and a **seasonal naive** baseline,
2. a **moving-average** or **exponential-smoothing** baseline,
3. at least one **ARIMA-family** model (ARIMA / SARIMAX with Fourier terms and/or lagged drivers),
4. at least one **machine-learning** model on lag features (Ridge / Random Forest / Gradient Boosting / XGBoost) predicting **returns**,
5. **your own idea** (e.g. a blend, a sliding window for the structural break, a regime indicator, a deep model, a threshold feature for temperature).

**Rules of thumb (from the lessons):** forecast log-returns or log prices; every feature lagged ≥ `h`; any driver you use must be **known at the origin** (lag it by ≥ horizon); fit scalers/imputers *inside* the model function (on `train` only); keep the model function deterministic (fix seeds).
You can test a model on a single origin quickly: `model(d.iloc[:500], 4, d[["festival_flag"]].iloc[500:504])`.

In [6]:
# TODO 5: write your feature function(s) and models, then register them in MODELS
# def features(ext, h): ...
# def model_seasonal_naive(train, h, fut): ...
# def model_arimax(train, h, fut): ...
# def model_ridge(train, h, fut): ...
# MODELS.update({"seasonal naive": model_seasonal_naive, ...})
# YOUR CODE HERE
print("registered models:", list(MODELS))

registered models: ['naive']


## 6 · Compare on the development period and select

The cell below runs every model in `MODELS` for each commodity over the dev origins and prints a scoreboard. *(Runtime grows with your models; ARIMAX and XGBoost refits take a few minutes in total. Start with one commodity while debugging.)*

**TODO 6**: (a) Inspect the scoreboards: which models beat naive, at which horizons? (b) Try at least one **blend** with `add_blend`. (c) Choose, for each commodity **and horizon**, the model you will carry to the test (using the dev scores only) and store it in `SELECTED`.

In [7]:
dev_wf = {}
for c, d in data.items():
    o_start = int(np.argmax(d.index >= "2022-01-01"))
    origins = range(o_start, len(d) - max(HORIZONS) + 1, 4)
    dev_wf[c] = run_walk_forward(d, MODELS, origins)
    print(f"\n=== {c.upper()}: development period, MAE (lower is better) ===")
    display(scoreboard(dev_wf[c]))

# TODO 6b: blends, e.g.  dev_wf[c] = add_blend(dev_wf[c], ["XGBoost", "naive"], "Blend (XGBoost + naive)")
# TODO 6c: pick a model per commodity and horizon, using the dev scoreboards ONLY, e.g.
SELECTED = {}      # {"beans": {1: "ARIMAX", 4: "...", 8: "...", 12: "..."}, "potato": {...}}

23 origins x 4 horizons x 1 models = 92 forecasts

=== BEANS: development period, MAE (lower is better) ===


,h=1,h=4,h=8,h=12,all,skill h=1,skill h=4,skill h=8,skill h=12,skill all
model,,,,,,,,,,
naive,13.123,28.363,28.797,34.398,26.17,0.0,0.0,0.0,0.0,0.0


23 origins x 4 horizons x 1 models = 92 forecasts

=== POTATO: development period, MAE (lower is better) ===


,h=1,h=4,h=8,h=12,all,skill h=1,skill h=4,skill h=8,skill h=12,skill all
model,,,,,,,,,,
naive,2.689,5.363,7.041,8.555,5.912,0.0,0.0,0.0,0.0,0.0


✍️ **Selection rationale** (why these models for these horizons? did anything surprise you?):

- …

## 7 · The one-shot test (locked)

The hold-out file is opened **only here**, **once**, after your models are final. Set `RUN_FINAL = True` when, and only when, you have:
- [ ] finished all modelling and selection on the dev period,
- [ ] filled `SELECTED` for every commodity and horizon,
- [ ] re-read the leakage checklist in `PROJECT_BRIEF.md`.

Peeking at the hold-out and then changing the models invalidates the project's central claim. If you *do* change something afterwards, **say so in your report**.

In [8]:
RUN_FINAL = False

if RUN_FINAL:
    hold = pd.read_csv("data/project_market_holdout_2024.csv", parse_dates=["date"])
    test_wf = {}
    for c, d_tr in data.items():
        d_all = pd.concat([d_tr, hold[hold.commodity == c].drop(columns="commodity").set_index("date")]); d_all.index.freq = "W-SUN"
        o_start = int(np.argmax(d_all.index >= "2024-01-01"))
        origins = range(o_start, len(d_all) - max(HORIZONS) + 1, 4)
        test_wf[c] = run_walk_forward(d_all, MODELS, origins)          # (re-add any blends with add_blend before scoring)
        print(f"\n=== {c.upper()}: 2024 hold-out (information: the models you SELECTED are the ones that count) ===")
        display(scoreboard(test_wf[c]))
else:
    print("🔒 Hold-out locked. Set RUN_FINAL = True when you are ready.")

🔒 Hold-out locked. Set RUN_FINAL = True when you are ready.


## 8 · Interpret and write the model card

**TODO 8**: Using `REPORT_TEMPLATE.md`, write (≈ 1–2 pages) a model card covering:
1. **Problem & data** (what you forecast, for whom, what was cleaned).
2. **Method** (models, features, evaluation protocol, how leakage was prevented).
3. **Results** (dev scoreboard, hold-out scoreboard for the *selected* models, skill vs naive, by horizon and commodity).
4. **Uncertainty** (at least one prediction-interval method and its coverage; see `empirical_interval` and `coverage`).
5. **Failure analysis** (where and why does it fail? look at the largest errors, which weeks?).
6. **Recommendation, limitations and next steps.**

Optional but impressive: feature importance, a global model across both commodities, a deep-learning model with a seed ensemble, a regime-aware model for the structural break.

In [9]:
# TODO 8: failure analysis, intervals, figures for your report
# YOUR CODE HERE

---
### ✅ Final checklist
- [ ] The notebook runs top to bottom (with `RUN_FINAL = True`).
- [ ] No feature uses information newer than `origin`; any driver is lagged ≥ horizon; `arrivals_tonnes` is not used.
- [ ] ≥ 5 models, ≥ 2 baselines, ≥ 1 ARIMA-family, ≥ 1 ML model.
- [ ] Selection made on the dev period only; hold-out evaluated once.
- [ ] Metrics reported by horizon *and* commodity, with skill vs naive.
- [ ] Uncertainty quantified; failures analysed; limitations stated honestly.